In [0]:
# %sql
# CREATE SCHEMA IF NOT EXISTS ecommerce.GOLD;

## Corresponding SQL query 

In [0]:
# %sql
# SELECT
#   A.ORDER_DATE_UPDATED,
#   COUNT(DISTINCT A.ORDER_ID) AS TOTAL_ORDERS,
#   SUM(
#     CASE
#       WHEN A.COMPLETED_ORDERS = 1 THEN 1
#       ELSE 0
#     END
#   ) AS COMPLETED_ORDERS,
#   SUM(
#     CASE
#       WHEN A.CANCELLED_ORDERS = 1 THEN 1
#       ELSE 0
#     END
#   ) AS CANCELLED_ORDERS,
#   SUM(
#     CASE
#       WHEN A.RETURNED_ORDERS = 1 THEN 1
#       ELSE 0
#     END
#   ) AS RETURNED_ORDERS,
#   SUM(
#     CASE
#       WHEN A.ORDER_STATUS = 'DELIVERED' THEN B.GROSS_AMOUNT
#       ELSE 0
#     END
#   ) AS GROSS_SALES,
#   SUM(
#     CASE
#       WHEN A.ORDER_STATUS = 'DELIVERED' THEN B.DISCOUNT_AMOUNT
#       ELSE 0
#     END
#   ) AS DISCOUNT_SALES,
#   SUM(
#     CASE
#       WHEN A.ORDER_STATUS = 'DELIVERED' THEN B.NET_AMOUNT
#       ELSE 0
#     END
#   ) AS NET_SALES,
#   SUM(
#     CASE
#       WHEN A.ORDER_STATUS = 'DELIVERED' THEN B.GROSS_AMOUNT
#       ELSE 0
#     END
#   )
#     / NULLIF(
#       SUM(
#         CASE
#           WHEN A.COMPLETED_ORDERS = 1 THEN 1
#           ELSE 0
#         END
#       ),
#       0
#     ) AS AVERAGE_ORDER_VALUE
# FROM
#   ECOMMERCE.SILVER.orders A
#     INNER JOIN ECOMMERCE.SILVER.order_items B
#       ON A.ORDER_ID = B.ORDER_ID
# GROUP BY
#   1
# ORDER BY
#   1
# LIMIT 10;

## Loading the libraries 

In [0]:
from pyspark.sql.functions import *

In [0]:

A = spark.table("ecommerce.silver.orders")
B = spark.table("ecommerce.silver.order_items")

In [0]:
daily_sales_df = (
    A.alias("A")
    .join(B.alias("B"), col("A.ORDER_ID") == col("B.ORDER_ID"), "inner")
    .groupBy(col("A.ORDER_DATE_UPDATED"))
    .agg(
        countDistinct(col("A.ORDER_ID")).alias("TOTAL_ORDERS"),
        sum(when(col("A.COMPLETED_ORDERS") == 1, 1).otherwise(0)).alias(
            "COMPLETED_ORDERS"
        ),
        sum(when(col("A.CANCELLED_ORDERS") == 1, 1).otherwise(0)).alias(
            "CANCELLED_ORDERS"
        ),
        sum(when(col("A.RETURNED_ORDERS") == 1, 1).otherwise(0)).alias(
            "RETURNED_ORDERS"
        ),
        sum(
            when(col("A.ORDER_STATUS") == "DELIVERED", col("B.GROSS_AMOUNT")).otherwise(
                0
            )
        ).alias("GROSS_SALES"),
        sum(
            when(
                col("A.ORDER_STATUS") == "DELIVERED", col("B.DISCOUNT_AMOUNT")
            ).otherwise(0)
        ).alias("DISCOUNT_SALES"),
        sum(
            when(col("A.ORDER_STATUS") == "DELIVERED", col("B.NET_AMOUNT")).otherwise(0)
        ).alias("NET_SALES"),
        (
            sum(
                when(
                    col("A.ORDER_STATUS") == "DELIVERED", col("B.GROSS_AMOUNT")
                ).otherwise(0)
            )
            / when(
                sum(when(col("A.COMPLETED_ORDERS") == 1, 1).otherwise(0)) != 0,
                sum(when(col("A.COMPLETED_ORDERS") == 1, 1).otherwise(0)),
            )
        ).alias("AVERAGE_ORDER_VALUE"),
    )
    .orderBy(col("ORDER_DATE_UPDATED"))
)

In [0]:
display(daily_sales_df)

In [0]:
daily_sales_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.gold.daily_sales_table"
)

## Business question 1: What is today's revenue

In [0]:
today_revenue = (
    daily_sales_df.orderBy(col("ORDER_DATE_UPDATED").desc())
    .select("NET_SALES")
    .first()[0]
)

print("Today's Revenue:", today_revenue)



## Business Question 2: Whatis the average order value

In [0]:
today_aov = (
    daily_sales_df.orderBy(col("ORDER_DATE_UPDATED").desc())
    .select("AVERAGE_ORDER_VALUE")
    .first()[0]
)

print("Today's Average Order Value:", today_aov)

## Business Question 3: Which day generated the highest revenue

In [0]:
highest_revenue_day = (
    daily_sales_df.orderBy(col("NET_SALES").desc())
    .select("ORDER_DATE_UPDATED", "NET_SALES")
    .first()
)

print("Highest Revenue Day:", highest_revenue_day["ORDER_DATE_UPDATED"])
print("Revenue:", highest_revenue_day["NET_SALES"])

## Business Question 4: What is the cancellation rate 

In [0]:
cancellation_rate = daily_sales_df.agg(
    (sum("CANCELLED_ORDERS") / sum("TOTAL_ORDERS") ).alias("CANCELLATION_RATE")
).first()[0]
print("Cancellation Rate:", cancellation_rate)